## AITechnicalTutor V2 - Added unit test generator
**Intro**

An AITechnicalTutor that helps with Python, Swift, and Java programming languages in a clear, simple, fun, and engaging way.

**Features**
- Uses a Gradio UI chat interface
- Streams responses
- Switch between models. 
- Tool calling, to generate diagrams, work with the file system, and generate unit tests.

**Tools**
- Create diagrams to visualize flowcharts, sequences, and class structures
- Work with files to read, write, and organize your code files
- Generate unit tests

**Feel free to ask anything like:**
- How do loops work in Python?
- What packages are in my workspace's requirements.txt
- Add this comment to the first line of `scraper.py` file: "This script scrapes data from the web"
- Show me a diagram for an auth flow in Python?
- Generate unit tests for `main.py`



In [ ]:
#Imports
import os
import tempfile
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

Helper classes:

DiagramGenerator

In [ ]:
class DiagramGenerator:
    @staticmethod
    def generate(diagram_type: str, description: str) -> dict:
        mermaid_templates = {
            "flowchart": lambda desc: f"flowchart TD\n{desc}",
            "sequence": lambda desc: f"sequenceDiagram\n{desc}",
            "class": lambda desc: f"classDiagram\n{desc}",
        }
        if diagram_type not in mermaid_templates:
            return {"error": f"Unsupported diagram type: {diagram_type}"}
        mermaid_code = mermaid_templates[diagram_type](description)
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mmd", mode="w") as f:
            f.write(mermaid_code)
            file_path = f.name
        return {"mermaid_code": mermaid_code, "file_path": file_path}


FileSystemTool

In [ ]:

class FileSystemTool:
    @staticmethod
    def operate(action: str, file_path: str = None, content: str = None, directory: str = None) -> dict:
        try:
            if action == "read" and file_path:
                with open(file_path, "r") as f:
                    return {"content": f.read()}
            elif action == "write" and file_path and content is not None:
                with open(file_path, "w") as f:
                    f.write(content)
                return {"status": f"Written to {file_path}"}
            elif action == "list" and directory:
                files = os.listdir(directory)
                return {"files": files}
            else:
                return {"error": "Invalid parameters"}
        except Exception as e:
            return {"error": str(e)}



UnitTestGenerator

In [ ]:
class UnitTestGenerator:
    @staticmethod
    def detect_language(code: str) -> str:
        if "def " in code and ":" in code:
            return "python"
        if "public class" in code or "System.out.println" in code:
            return "java"
        if "func " in code or "import Swift" in code:
            return "swift"
        return "unknown"

    @staticmethod
    def generate(client, model, code: str, language: str = None, framework: str = None):

        if not code:
            return {"error": "No code provided"}

        if not language:
            language = UnitTestGenerator.detect_language(code)

        framework_map = {
            "python": "pytest",
            "java": "JUnit",
            "swift": "XCTest"
        }

        if not framework:
            framework = framework_map.get(language, "appropriate framework")

        prompt = f"""
            You are an expert software test engineer.
            Generate high quality {framework} unit tests for the following {language} code.
            
            Rules:
            - Output ONLY the unit tests
            - No explanations
            - No markdown
            - Tests must be runnable
            - Cover edge cases
            
            Code:
            {code}
        """

        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": prompt}]
        )

        tests = response.choices[0].message.content

        return {
            "tests": tests,
            "language": language,
            "framework": framework
        }

ToolSpecFactory

In [ ]:
class ToolSpecFactory:
    @staticmethod
    def diagram_generator_spec():
        return {
            "name": "generate_diagram",
            "description": (
                "Generate a diagram (flowchart, sequence diagram, or class diagram) from a user’s description of logic, code, or process. "
                "You must always specify the diagram_type (\"flowchart\", \"sequence\", or \"class\") and a description."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "diagram_type": {"type": "string"},
                    "description": {"type": "string"}
                },
                "required": ["diagram_type", "description"]
            }
        }

    @staticmethod
    def file_system_tool_spec():
        return {
            "name": "file_system_tool",
            "description": (
                "Read, write, or list files in the workspace. "
                "You must always specify the 'action' parameter: 'read', 'write', or 'list'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "action": {"type": "string", "enum": ["read", "write", "list"]},
                    "file_path": {"type": "string"},
                    "content": {"type": "string"},
                    "directory": {"type": "string"}
                },
                "required": ["action"]
            }
        }
    
    @staticmethod
    def unit_test_generator_spec():
        return {
            "name": "generate_unit_tests",
            "description": (
                "Generate unit tests for code. Code may come from user input or from files. "
                "The generated tests can optionally be written to a file."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {"type": "string"},
                    "language": {"type": "string"},
                    "framework": {"type": "string"},
                    "write_to_file": {"type": "boolean"},
                    "output_path": {"type": "string"}
                },
                "required": ["code"]
            }
    }


AITechnicalTutor class with unit test tool

In [ ]:
class AITechnicalTutor:
    def __init__(self):
        load_dotenv(override=True)
        self.openai_api_key = os.getenv('OPENAI_API_KEY')
        self.anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
        self.google_api_key = os.getenv('GOOGLE_API_KEY')
        self.system_prompt = """
            You are an expert in Swift, Python, and Java programming languages.
            You answer technical questions related to Swift, Python, and Java languages in a clear manner 
            for teenagers learning to program for the first time. Your response is clear, fun, engaging, indepth, and without fluff. 
            You can also generate diagrams, read and write files, and generate unit tests for code. 
            If the user asks for a diagram or about files, use the appropriate tool.
            If a user asks for unit tests, use the generate_unit_tests tool. 
            If code is stored in a file, read the file using the file_system_tool first."
        """
        self.tools = [
            {"type": "function", "function": ToolSpecFactory.diagram_generator_spec()},
            {"type": "function", "function": ToolSpecFactory.file_system_tool_spec()},
            {"type": "function", "function": ToolSpecFactory.unit_test_generator_spec()}
        ]
        self.clients = {
            "openai": OpenAI(),
            "claude": OpenAI(base_url="https://api.anthropic.com/v1/", api_key=self.anthropic_api_key),
            "gemini": OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=self.google_api_key)
        }
        self.models = {
            "openai": "gpt-4.1-mini",
            "claude": "claude-haiku-4-5",
            "gemini": "gemini-2.5-flash-lite"
        }

    def handle_tool_calls(self, tool_calls, provider):
        assistant_message = {
            "role": "assistant",
            "content": "",
            "tool_calls": []
        }
        tool_responses = []
        for tc in tool_calls.values():
            try:
                args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            except:
                args = {}

            tool_call = {
                "id": tc["id"],
                "type": "function",
                "function": {
                    "name": tc["name"],
                    "arguments": tc["arguments"]
                }
            }
            assistant_message["tool_calls"].append(tool_call)

            if tc["name"] == "generate_diagram":
                if "diagram_type" in args and "description" in args:
                    result = DiagramGenerator.generate(**args)
                else:
                    result = {"error": "Missing required arguments: 'diagram_type', 'description'"}
            elif tc["name"] == "file_system_tool":
                if "action" in args:
                    result = FileSystemTool.operate(**args)
                else:
                    result = {"error": "Missing required argument: 'action'"}
            elif tc["name"] == "generate_unit_tests":
                result = UnitTestGenerator.generate(
                    client= self.clients[provider], 
                    model= self.models[provider], 
                    code=args.get("code"),
                    language=args.get("language"),
                    framework=args.get("framework")
                )

                if args.get("write_to_file") and "tests" in result:
                    output_path = args.get("output_path")

                    if not output_path:
                        output_path = "generated_tests.py"

                    try:
                        with open(output_path, "w") as f:
                            f.write(result["tests"])
                        result["file_written"] = output_path
                    except Exception as e:
                        result["error"] = str(e)
                        
            else:
                result = {"error": f"Unknown tool {tc['name']}"}

            tool_responses.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": json.dumps(result)
            })
        return assistant_message, tool_responses

    def build_messages(self, message, history):
        return (
            [{"role": "system", "content": self.system_prompt}]
            + [{"role": h["role"], "content": h["content"]} for h in history]
            + [{"role": "user", "content": message}]
        )

    def stream_response(self, messages, client, model):
        return client.chat.completions.create(
            model=model,
            tools=self.tools,
            messages=messages,
            stream=True
        )

    def collect_stream_chunks(self, stream):
        response_text = ""
        tool_calls = {}
        finish_reason = None

        for chunk in stream:
            choice = chunk.choices[0]
            delta = choice.delta
            finish_reason = choice.finish_reason

            if getattr(delta, "content", None):
                response_text += delta.content
                yield response_text, tool_calls, finish_reason

            if getattr(delta, "tool_calls", None):
                for tc in delta.tool_calls:
                    idx = tc.index
                    if idx not in tool_calls:
                        tool_calls[idx] = {
                            "id": tc.id,
                            "name": "",
                            "arguments": ""
                        }
                    if tc.function:
                        if tc.function.name:
                            tool_calls[idx]["name"] = tc.function.name
                        if tc.function.arguments:
                            tool_calls[idx]["arguments"] += tc.function.arguments
        yield response_text, tool_calls, finish_reason


    def chat(self, message, history, provider):
        client = self.clients[provider]
        model = self.models[provider]
        messages = self.build_messages(message, history)
        while True:
            response = client.chat.completions.create(
                model=model,
                tools=self.tools,
                messages=messages
            )
            choice = response.choices[0]
            finish_reason = choice.finish_reason
            if finish_reason == "tool_calls":
                tool_calls = []
                for tc in choice.message.tool_calls:
                    tool_calls.append({
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    })
                assistant_message = {
                    "role": "assistant",
                    "content": "",
                    "tool_calls": tool_calls
                }
                tool_responses = self._handle_tool_calls(tool_calls)
                messages.append(assistant_message)
                messages.extend(tool_responses)
                continue
            return choice.message.content

    def stream_chat(self, message, history, provider):
        client = self.clients[provider]
        model = self.models[provider]
        messages = self.build_messages(message, history)
        while True:
            stream = self.stream_response(messages, client, model)
            response_text = ""
            tool_calls = {}
            finish_reason = None

            for response_text, tool_calls, finish_reason in self.collect_stream_chunks(stream):
                if response_text:
                    yield response_text

            if finish_reason == "stop":
                break

            if finish_reason == "tool_calls":
                assistant_message, tool_responses = self.handle_tool_calls(tool_calls, provider)
                messages.append(assistant_message)
                messages.extend(tool_responses)
                continue
            
    def launch_gradio(self, stream=False):
        def chat_wrapper(message, history, provider):
            if stream:
                yield from self.stream_chat(message, history, provider)
            else:
                return self.chat(message, history, provider)

        gr.ChatInterface(
            fn=chat_wrapper,
            type="messages",
            additional_inputs=gr.Dropdown(
                choices=["openai", "claude", "gemini"],
                value="openai",
                label="Model Provider"
            )
        ).launch()

Launch AITechnicalTutor

In [ ]:

tutor = AITechnicalTutor()
tutor.launch_gradio(stream=True)